# GNN-MARL Network Slicing — Training Notebook

Jalankan cell satu per satu dari atas ke bawah.

| Hardware | PPO (1M steps) | DQN (200K steps) |
|---|---|---|
| Colab T4   | ~45 menit/algo | ~2 jam/algo  |
| Colab A100 | ~15 menit/algo | ~45 menit/algo |

> **Sebelum mulai:** Runtime → Change runtime type → **GPU** (T4 gratis, A100 Colab Pro)  
> Hasil disimpan ke Google Drive — aman kalau session disconnect.  
> Kalau reconnect: jalankan ulang semua cell dari atas; cell yang sudah selesai **auto-skip**.

## Cell 1 — Mount Drive + Clone Repo

In [ ]:
import os, sys, subprocess, time, csv, glob

# ── GANTI INI ────────────────────────────────────────────────────────────
REPO_URL  = "https://github.com/YOUR_USERNAME/gnn-marl-network-slicing.git"
DRIVE_DIR = "/content/drive/MyDrive/gnn-marl-thesis"
# ─────────────────────────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists(f"{DRIVE_DIR}/.git"):
    subprocess.run(["git", "clone", REPO_URL, DRIVE_DIR], check=True)
else:
    subprocess.run(["git", "-C", DRIVE_DIR, "pull"], check=True)

os.chdir(DRIVE_DIR)
print("CWD:", os.getcwd())

## Cell 2 — Install Dependencies

In [ ]:
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "torch==2.4.0", "--index-url", "https://download.pytorch.org/whl/cu124"
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "torch-geometric==2.8.0",
    "numpy==1.26.4", "scipy==1.13.0",
    "gymnasium==1.1.0", "pettingzoo==1.25.0", "pyyaml==6.0.2"
], check=True)

import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    raise RuntimeError("GPU not found! Runtime → Change runtime type → GPU")

## Cell 3 — Smoke Test (70 tests)

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-q", "--tb=short"],
    capture_output=True, text=True
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print(result.stderr[-500:])
    raise RuntimeError("Tests gagal — cek output")

## Cell 4 — Helper Functions

In [ ]:
def _csv_done(algo, backbone=None, seed=42, steps=None):
    """True kalau last step di CSV >= 90% target steps."""
    name = f"{algo}_{backbone}_seed{seed}.csv" if backbone else f"{algo}_seed{seed}.csv"
    path = f"results/logs/{name}"
    if not os.path.exists(path):
        return False
    with open(path) as f:
        rows = [r for r in f.read().splitlines() if r]
    if len(rows) < 2:
        return False
    try:
        last_step = int(rows[-1].split(',')[0])
    except (ValueError, IndexError):
        return False
    threshold = int(steps * 0.9) if steps else 900_000
    return last_step >= threshold


def train(algo, backbone=None, steps=1_000_000, seed=42):
    label = f"{algo}/{backbone}" if backbone else algo
    if _csv_done(algo, backbone, seed, steps):
        print(f"[SKIP]  {label} — sudah selesai (>= 90% steps)")
        return
    os.makedirs("results/logs", exist_ok=True)
    if backbone:
        cmd = [sys.executable, "training/train_proposed.py",
               "--algo", algo, "--backbone", backbone,
               "--steps", str(steps), "--seed", str(seed)]
    else:
        cmd = [sys.executable, "training/train_baselines.py",
               "--algo", algo, "--steps", str(steps), "--seed", str(seed)]
    print(f"\n{'='*55}\n  {label}  ({steps:,} steps)\n{'='*55}")
    t0 = time.time()
    subprocess.run(cmd, check=True)
    print(f"  Selesai dalam {(time.time()-t0)/60:.1f} menit")


def check_progress():
    log_dir = "results/logs"
    if not os.path.exists(log_dir):
        print("Belum ada logs."); return
    print(f"{'File':<45} {'Last Step':>12} {'Target':>10} {'%':>6} {'Last Reward':>12}")
    print("-" * 95)
    targets = {
        "gnn-mappo": 1_000_000, "gnn-madqn": 200_000,
        "ippo": 1_000_000, "central-ppo": 1_000_000,
        "idqn": 200_000, "central-dqn": 200_000,
    }
    for f in sorted(glob.glob(f"{log_dir}/*.csv")):
        with open(f) as fp:
            rows = list(csv.reader(fp))
        n = len(rows) - 1
        if n > 0:
            last = rows[-1]
            step = int(last[0]) if last[0].isdigit() else 0
            rew  = last[2] if len(last) > 2 else "-"
            algo_key = next((k for k in targets if k in f), None)
            target = targets.get(algo_key, 1_000_000)
            pct = f"{step/target*100:.0f}%"
            print(f"{os.path.basename(f):<45} {step:>12,} {target:>10,} {pct:>6} {rew:>12}")
        else:
            print(f"{os.path.basename(f):<45} {'(empty)':>12}")


def clear_partial_logs():
    """Hapus CSV yang belum selesai (partial training dari mesin lain)."""
    targets = {
        "gnn-mappo": 1_000_000, "gnn-madqn": 200_000,
        "ippo": 1_000_000, "central-ppo": 1_000_000,
        "idqn": 200_000, "central-dqn": 200_000,
    }
    for f in glob.glob("results/logs/*.csv"):
        with open(f) as fp:
            rows = [r for r in fp.read().splitlines() if r]
        if len(rows) < 2:
            os.remove(f); print(f"Deleted (empty): {f}"); continue
        try:
            last_step = int(rows[-1].split(',')[0])
        except (ValueError, IndexError):
            continue
        algo_key = next((k for k in targets if k in f), None)
        target = targets.get(algo_key, 1_000_000)
        if last_step < target * 0.9:
            os.remove(f)
            print(f"Deleted (partial {last_step:,}/{target:,}): {os.path.basename(f)}")
        else:
            print(f"Kept   (done     {last_step:,}/{target:,}): {os.path.basename(f)}")


print("Helper functions loaded.")

In [ ]:
# Jalankan ini kalau ada CSV partial dari training sebelumnya (laptop/server lain)
# Akan hapus CSV yang belum selesai, biarkan yang sudah done
clear_partial_logs()

## Cell 5 — PPO Training ⚡

Paling cepat. Estimasi T4: **~3 jam total** (4 algo × 45 menit).

In [ ]:
train("gnn-mappo", backbone="gat",  steps=1_000_000)  # GNN-MAPPO / GAT

In [ ]:
train("gnn-mappo", backbone="sage", steps=1_000_000)  # GNN-MAPPO / SAGE

In [ ]:
train("ippo",        steps=1_000_000)  # IPPO baseline (MLP, independent)

In [ ]:
train("central-ppo", steps=1_000_000)  # Central-PPO baseline (MLP, centralized)

## Cell 6 — DQN Training 🐢

CPU-bottlenecked (env simulation). 200K steps cukup untuk convergence comparison.

Estimasi T4: **~2 jam per algo**. Kalau khawatir timeout: jalankan 1 cell per sesi Colab.

In [ ]:
train("gnn-madqn", backbone="gat",  steps=200_000)  # GNN-MADQN / GAT

In [ ]:
train("gnn-madqn", backbone="sage", steps=200_000)  # GNN-MADQN / SAGE

In [ ]:
train("idqn",        steps=200_000)  # IDQN baseline (MLP, independent)

In [ ]:
train("central-dqn", steps=200_000)  # Central-DQN baseline (MLP, centralized)

## Cell 7 — Cek Progress (kapan saja)

In [ ]:
check_progress()

## Cell 8 — Evaluasi

Jalankan setelah semua training selesai. Butuh file `.pt` dari tiap algo.

In [ ]:
# Phase 2a: Convergence curves → results/figures/
subprocess.run([
    sys.executable, "evaluation/convergence_eval.py",
    "--log-dir", "results/logs"
], check=True)

In [ ]:
# Phase 2b: Network performance (throughput, latency, Jain's fairness)
subprocess.run([sys.executable, "evaluation/network_perf_eval.py"], check=True)

In [ ]:
# Phase 2c: Zero-shot generalization 5 → 10 → 20 gNB
subprocess.run([sys.executable, "evaluation/zero_shot_eval.py"], check=True)

In [ ]:
# Phase 3: Ablation (backbone / reward weights / depth / burstiness)
for ablation in ["backbone", "reward", "depth", "burstiness"]:
    print(f"\n=== Ablation: {ablation} ===")
    subprocess.run([
        sys.executable, "ablation/backbone_ablation.py",
        "--ablation", ablation, "--steps", "200000"
    ], check=True)

print("\nSemua evaluasi selesai! Hasil di folder results/")